In [ ]:
#import statements 

from pathlib import Path
import numpy as np
import pandas as pd
import pyreadstat
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor, XGBClassifier
from sklearn.metrics import roc_auc_score
from collections import Counter
from itertools import combinations


Preprocessing Functions

In [ ]:
def LFM_data_from_raw(stu_raw: pd.DataFrame, sch_raw: pd.DataFrame, country: str) -> pd.DataFrame:
    print("Loading student and school data...")
    print("Filtering student and school data for country...")
    stu = stu_raw[stu_raw["CNT"] == country].copy()
    sch = sch_raw[sch_raw["CNT"] == country].copy()

    keep_sch = [c for c in ["CNTSCHID","RATCMP1","RATCMP2","EDUSHORT"] if c in sch.columns]
    if keep_sch:
        sch = sch[["CNTSCHID"] + keep_sch[1:]] if "CNTSCHID" in sch.columns else sch

    print("Merging student + school data...")
    if "CNTSCHID" in stu.columns and "CNTSCHID" in sch.columns:
        df = stu.merge(sch, on="CNTSCHID", how="left")
    else:
        df = stu.copy()  # fall back if CNTSCHID missing
    return df

def process_data_beingbullied(df: pd.DataFrame, predictors: list, dv: str = "BEINGBULLIED") -> pd.DataFrame:
    print("Keeping predictors + DV, encoding gender...")
    cols = [c for c in predictors if c in df.columns]
    if dv not in df.columns:
        raise ValueError(f"DV '{dv}' not in dataframe.")
    keep = list(dict.fromkeys(cols + [dv] + (["CNTSCHID"] if "CNTSCHID" in df.columns else [])))
    df = df[keep].copy()

    # encode gender
    if "ST004D01T" in df.columns:
        df["ST004D01T"] = df["ST004D01T"].map({1:0, 2:1})

    # drop rows with missing DV
    df = df.dropna(subset=[dv])

    # X / y
    X = df.drop(columns=[dv, "CNTSCHID"], errors="ignore")
    print("Dropping columns with all missing values...")
    X = X.dropna(axis=1, how="all")

    print("Imputing missing values and scaling features...")
    imputer = SimpleImputer(strategy="median")
    scaler  = StandardScaler()
    X_imp   = pd.DataFrame(imputer.fit_transform(X), columns=X.columns, index=X.index)
    X_scl   = pd.DataFrame(scaler.fit_transform(X_imp), columns=X.columns, index=X.index)

    X_scl[dv] = df[dv].values
    return X_scl

def process_data_bullying_types(df: pd.DataFrame, predictors: list, RAW_ITEMS) -> pd.DataFrame:
    print("Keeping predictors + raw bullying items, encoding gender...")
    keep_pred = [c for c in predictors if c in df.columns]
    keep_raw  = [c for c in RAW_ITEMS if c in df.columns]
    keep_cols = list(dict.fromkeys(keep_pred + keep_raw + (["CNTSCHID"] if "CNTSCHID" in df.columns else [])))
    df = df[keep_cols].copy()

    if "ST004D01T" in df.columns:
        df["ST004D01T"] = df["ST004D01T"].map({1:0, 2:1})

    print("Creating binary DVs for 4 bullying types...")
    def _bin4(s: pd.Series) -> pd.Series:
        if s.dropna().max() is not None and s.dropna().max() >= 4:
            return s.isin([3,4]).astype(float)
        return s.isin([2,3]).astype(float)

    dv_map = {}
    if "ST038Q04NA" in df.columns: dv_map["BULLY_VERBAL"]     = _bin4(df["ST038Q04NA"])
    if "ST038Q05NA" in df.columns: dv_map["BULLY_THREAT"]     = _bin4(df["ST038Q05NA"])
    if "ST038Q08NA" in df.columns: dv_map["BULLY_RELATIONAL"] = _bin4(df["ST038Q08NA"])
    if {"ST038Q06NA","ST038Q07NA"}.issubset(df.columns):
        dv_map["BULLY_PHYSICAL"] = (_bin4(df["ST038Q06NA"]).astype(bool) | _bin4(df["ST038Q07NA"]).astype(bool)).astype(float)

    if not dv_map:
        raise ValueError("No bullying type items found to create DVs.")

    dv_df = pd.DataFrame(dv_map, index=df.index)

    print("Dropping columns with all missing values...")
    X = df.drop(columns=RAW_ITEMS + ["CNTSCHID"], errors="ignore").dropna(axis=1, how="all")

    print("Imputing missing values and scaling features...")
    imputer = SimpleImputer(strategy="median")
    scaler  = StandardScaler()
    X_imp   = pd.DataFrame(imputer.fit_transform(X), columns=X.columns, index=X.index)
    X_scl   = pd.DataFrame(scaler.fit_transform(X_imp), columns=X.columns, index=X.index)

    final_df = pd.concat([X_scl, dv_df], axis=1)
    return final_df


Predictor Analysis Functions

In [ ]:
def best_xgb_reg():
    return XGBRegressor(
        n_estimators=300, max_depth=4, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        reg_lambda=4, reg_alpha=0.6, min_child_weight=5,
        objective="reg:squarederror", gamma=0,
        random_state=42, verbosity=0, n_jobs=-1
    )

def best_xgb_clf():
    return XGBClassifier(
        n_estimators=300, max_depth=4, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        reg_lambda=4, reg_alpha=0.6, min_child_weight=5,
        objective="binary:logistic", gamma=0, eval_metric="auc",
        random_state=42, verbosity=0, n_jobs=-1
    )

def top10_beingbullied(df_proc: pd.DataFrame, country_label: str) -> pd.DataFrame:
    dv = "BEINGBULLIED"
    if dv not in df_proc.columns:
        raise ValueError(f"{country_label}: '{dv}' not found after processing.")

    X = df_proc.drop(columns=[dv], errors="ignore")
    y = df_proc[dv].values

    # Need at least some non-NaN variance to train
    if len(X.columns) == 0 or np.isclose(X.var(numeric_only=True), 0).all():
        raise ValueError(f"{country_label}: no usable predictors after cleaning.")

    model = best_xgb_reg().fit(X, y)
    imp = pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=False).head(10)

    out = (imp.reset_index()
              .rename(columns={"index":"Feature", 0:"Importance"})
              .assign(Country=country_label, Type="BEINGBULLIED"))
    out["Rank"] = np.arange(1, len(out)+1)
    return out[["Country","Type","Rank","Feature","Importance"]]


def top10_bullying_types(df_proc: pd.DataFrame, country_label: str, DVLIST) -> pd.DataFrame:
    X = df_proc.drop(columns=DVLIST, errors="ignore")
    if X.shape[1] == 0:
        raise ValueError(f"{country_label}: no predictors available.")

    rows = []
    for dv in [c for c in DVLIST if c in df_proc.columns]:
        y = df_proc[dv].dropna().astype(int)
        mask = df_proc[dv].notna()
        Xy = X.loc[mask]

        # skip if only one class
        if y.nunique() < 2:
            print(f"{country_label} — {dv.replace('BULLY_','').title()}: skipped (only one class).")
            continue

        model = best_xgb_clf().fit(Xy, y)
        auc = roc_auc_score(y, model.predict_proba(Xy)[:,1])

        imp = (pd.Series(model.feature_importances_, index=Xy.columns)
                 .sort_values(ascending=False)
                 .head(10))

        tmp = (imp.reset_index()
                 .rename(columns={"index":"Feature", 0:"Importance"})
                 .assign(Country=country_label, Type=dv, AUC=auc))
        tmp["Rank"] = np.arange(1, len(tmp)+1)
        rows.append(tmp[["Country","Type","AUC","Rank","Feature","Importance"]])

    if not rows:
        return pd.DataFrame(columns=["Country","Type","AUC","Rank","Feature","Importance"])
    return pd.concat(rows, ignore_index=True)


Data Preprocessing and Predictor Analysis

In [ ]:
STU_PATH = Path("../data/raw/Student Data.sav")
SCH_PATH = Path("../data/raw/School Data.sav")

PROC_DIR   = Path("../data/processed")
RESULT_DIR = Path("../results")
PROC_DIR.mkdir(parents=True, exist_ok=True)
RESULT_DIR.mkdir(parents=True, exist_ok=True)

PREDICTORS = [
    #Individual-level Predictors
    "ST004D01T", #Gender
    "AGE", #Age
    "GRADE", #Grade
    "BSMJ", #Expected Occupational Status
    "JOYREAD", #Joy of Reading
    "SCREADCOMP", #Reading Self-Concept: Competence
    "SCREADDIFF", #Reading Self-Concept: Difficulty
    "COMPETE", #Competitiveness
    "WORKMAST", #Work Mastery Orientation
    "GFOFAIL", # General Fear of Failure
    "EUDMO", #Sense of Meaning in Life (Eudaimonia)  
    "RESILIENCE", #Resilience
    "MASTGOAL", #Mastery Goal Orientation
    "ST185Q01HA", #Does life has meaning/purpose 
    "ST184Q01HA", #Growth Mindset
    "SWBP", #Well Being 
    "PV1MATH", #Math Performance
    "PV1READ", #Reading Performance
    "PV1SCIE", #Science Performance
    "ST207Q01HA", #Irritation feeling when other students are bullied
    "ST207Q02HA", #Feeling of whether it's good or not to help students who can't defend themselves
    "ST207Q03HA", #Is it wrong to join in bullying others
    "ST207Q04HA", #How you feel when you see other students being bullied
    "ST207Q05HA", #Liking when you see other students being bullied

    #Proximal-level Predictors
    "REPEAT", #Grade Repetition History
    "UNDREM", #Meta-cognition: Understanding & Remembering 
    "METASUM", #Meta-cognition: Summarizing
    "METASPAM", #Meta-cognition: Assessing Credibility

    #Microsystem-Level Factors (Family, Peers, & School CLimate)
    "EMOSUPS", #Parental Emotional Support
    "DURECEC", #Duration in Early Childhood Education and Care
    "BELONG", #School Belonging
    "PERCOMP", #Perceived School Competitiveness 
    "PERCOOP", #Perceived School Cooperation
    "ATTLNACT", #Attitudes Towards Learning Activities
    "DISCLIMA", #Disciplinary Climate (Language Lessons)
    "TEACHSUP", #Teacher Support (Language Lessons)
    "DIRINS", #Teacher-Directed Instruction 
    "PERFEED", #Perceived Feedback from Teachers
    "STIMREAD", #Teacher's Stimulation of Reading Engagement
    "ADAPTIVITY", #Adapation of Instruction
    "TEACHINT", #Perceived Teacher Interest 
    "ST206Q01HA", #How students value cooperation 
    "PA006Q09TA" #School Climate

    #Macrosystem/Exosystem-level Predictors
    "ESCS", #Family Socioeconomic Status(Index)
    "EDUSHORT", #Shortage of Educational Resources
    "RATCMP1", #Number of Computers per Student
    "RATCMP2", #Percentage of Computers Connected to the Internet                    
]

RAW_TYPE_ITEMS = ["ST038Q04NA","ST038Q05NA","ST038Q06NA","ST038Q07NA","ST038Q08NA"]
DV_TYPES = ["BULLY_VERBAL","BULLY_THREAT","BULLY_PHYSICAL","BULLY_RELATIONAL"]

error_log = []
def log_error(iso3, stage, err):
    msg = f"{iso3} | {stage} | {type(err).__name__}: {err}"
    print("!!", msg)
    error_log.append({"country": iso3, "stage": stage, "error": str(err)})


print("Preloading raw PISA files once...")
stu_raw, _ = pyreadstat.read_sav(STU_PATH, apply_value_formats=False)
sch_raw, _ = pyreadstat.read_sav(SCH_PATH, apply_value_formats=False)
print("Preload complete.")

# Figure out all 2018 ISO3 country codes directly from the data
ALL_ISO3 = sorted(stu_raw["CNT"].dropna().unique().tolist())

# Containers for outputs
rows_being = []
rows_types = []
prev_rows  = []

for iso3 in ALL_ISO3:
    try:
        print(f"\nPreprocessing PISA data for {iso3}...")
        df_merged = LFM_data_from_raw(stu_raw, sch_raw, iso3)

        try:
            df_being = process_data_beingbullied(df_merged, PREDICTORS, dv="BEINGBULLIED")
            # save processed per-country
            outp = PROC_DIR / f"{iso3}_DVBEING_data.pkl"
            df_being.to_pickle(outp)
            print(f"Saved processed BEINGBULLIED → {outp} (shape: {df_being.shape})")

            # prevalence proxy = mean of DV (works if 0/1 or standardized index)
            prev_rows.append({"Country": iso3, "Type": "BEINGBULLIED", "Prevalence": float(np.nanmean(df_being["BEINGBULLIED"]))})

            # top10
            try:
                top10 = top10_beingbullied(df_being, iso3)
                rows_being.append(top10)
                # pretty print
                print(f"Top 10 XGB predictors of BEINGBULLIED ( {iso3} ):")
                for _, r in top10.iterrows():
                    print(f" {int(r['Rank']):2d}. {r['Feature']:<12s}: {r['Importance']:.4f}")
            except Exception as e:
                log_error(iso3, "top10_beingbullied", e)

        except Exception as e:
            log_error(iso3, "process_data_beingbullied", e)

        try:
            df_types = process_data_bullying_types(df_merged, PREDICTORS, RAW_TYPE_ITEMS)
            outp2 = PROC_DIR / f"{iso3}_bully_types_data.pkl"
            df_types.to_pickle(outp2)
            print(f"Saved processed BullyTypes → {outp2} (shape: {df_types.shape})")

            # Prevalence for each binary type (mean of 0/1)
            for dv in [c for c in DV_TYPES if c in df_types.columns]:
                prev_rows.append({"Country": iso3, "Type": dv, "Prevalence": float(np.nanmean(df_types[dv]))})

            # top10 per type
            try:
                top_types = top10_bullying_types(df_types, iso3, DV_TYPES)
                rows_types.append(top_types)
                # optional pretty print (short)
                if not top_types.empty:
                    for dv in top_types["Type"].unique():
                        sub = top_types[top_types["Type"]==dv]
                        auc = sub["AUC"].iloc[0]
                        print(f"{iso3} — {dv.replace('BULLY_','').title()} (ROC-AUC {auc:.3f})")
                        for _, r in sub.iterrows():
                            print(f" {int(r['Rank']):2d}. {r['Feature']:<12s}: {r['Importance']:.4f}")
            except Exception as e:
                log_error(iso3, "top10_bullying_types", e)

        except Exception as e:
            log_error(iso3, "process_data_bullying_types", e)

    except Exception as e:
        log_error(iso3, "LFM_data_from_raw", e)

df_being_all = pd.concat(rows_being, ignore_index=True) if rows_being else pd.DataFrame(columns=["Country","Type","Rank","Feature","Importance"])
df_types_all = pd.concat(rows_types, ignore_index=True) if rows_types else pd.DataFrame(columns=["Country","Type","AUC","Rank","Feature","Importance"])
df_prev      = pd.DataFrame(prev_rows)

df_being_all.to_csv(RESULT_DIR / "beingbullied_top10.csv", index=False)
df_types_all.to_csv(RESULT_DIR / "bullying_types_top10.csv", index=False)
df_prev.to_csv(RESULT_DIR / "bullying_prevalence.csv", index=False)

pd.DataFrame(error_log).to_csv(RESULT_DIR / "errors_log.csv", index=False)
print("\nSaved:")
print(" - results/beingbullied_top10.csv")
print(" - results/bullying_types_top10.csv")
print(" - results/bullying_prevalence.csv")
print(" - results/errors_log.csv")

def add_points(df_long, has_auc=False):
    if df_long.empty: 
        return df_long
    pts_map = {1:10,2:9,3:8,4:7,5:6,6:5,7:4,8:3,9:2,10:1}
    out = df_long.copy()
    out["Points"] = out["Rank"].map(pts_map).fillna(0).astype(int)
    return out

being_pts = add_points(df_being_all, has_auc=False)
types_pts = add_points(df_types_all, has_auc=True)

# Global totals by feature (separately for BEINGBULLIED and for each Type)
agg_being = (being_pts.groupby("Feature", as_index=False)["Points"].sum()
                        .sort_values("Points", ascending=False))
agg_types = (types_pts.groupby(["Type","Feature"], as_index=False)["Points"].sum()
                        .sort_values(["Type","Points"], ascending=[True, False]))

agg_being.to_csv(RESULT_DIR / "aggregate_points_BEINGBULLIED.csv", index=False)
agg_types.to_csv(RESULT_DIR / "aggregate_points_byType.csv", index=False)

print("\nSaved:")
print(" - results/aggregate_points_BEINGBULLIED.csv")
print(" - results/aggregate_points_byType.csv")

Similarity Analysis

In [ ]:
RESULT_DIR = Path("../results")
OUT_DIR = RESULT_DIR / "pattern_analysis"
OUT_DIR.mkdir(parents=True, exist_ok=True)

BEING10_CSV = RESULT_DIR / "beingbullied_top10.csv"       # Country,Type,Rank,Feature,Importance
TYPES10_CSV = RESULT_DIR / "bullying_types_top10.csv"     # Country,Type,AUC,Rank,Feature,Importance

OUTCOMES = ["BEINGBULLIED", "BULLY_VERBAL", "BULLY_THREAT", "BULLY_PHYSICAL", "BULLY_RELATIONAL"]

POINTS_MAP = {1:10,2:9,3:8,4:7,5:6,6:5,7:4,8:3,9:2,10:1}

PRED_TO_CATEGORY = {
    # Individual
    "ST004D01T":"Individual","AGE":"Individual","GRADE":"Individual","BSMJ":"Individual","JOYREAD":"Individual",
    "SCREADCOMP":"Individual","SCREADDIFF":"Individual","COMPETE":"Individual","WORKMAST":"Individual",
    "GFOFAIL":"Individual","EUDMO":"Individual","RESILIENCE":"Individual","MASTGOAL":"Individual",
    "PV1READ":"Individual","PV1MATH":"Individual","PV1SCIE":"Individual",

    # Proximal
    "REPEAT":"Proximal","UNDREM":"Proximal","METASUM":"Proximal","METASPAM":"Proximal",

    # Microsystem
    "EMOSUPS":"Microsystem","DURECEC":"Microsystem","BELONG":"Microsystem","PERCOMP":"Microsystem",
    "PERCOOP":"Microsystem","ATTLNACT":"Microsystem","DISCLIMA":"Microsystem","TEACHSUP":"Microsystem",
    "DIRINS":"Microsystem","PERFEED":"Microsystem","STIMREAD":"Microsystem","ADAPTIVITY":"Microsystem",
    "TEACHINT":"Microsystem","SWBP":"Microsystem",
    "ST206Q01HA":"Microsystem","ST207Q01HA":"Microsystem","ST207Q02HA":"Microsystem","ST207Q03HA":"Microsystem",
    "ST207Q04HA":"Microsystem","ST207Q05HA":"Microsystem",

    # Macro
    "ESCS":"Macro","EDUSHORT":"Macro","RATCMP1":"Macro","RATCMP2":"Macro",
}

def load_top10():
    df_being = pd.read_csv(BEING10_CSV)
    df_types = pd.read_csv(TYPES10_CSV)

    # Normalize columns
    for df in (df_being, df_types):
        df.columns = [c.strip().title() for c in df.columns]
        if "Feature" in df.columns:
            df.rename(columns={"Feature": "Predictor"}, inplace=True)

    # Defensive filter to rank<=10
    df_being = df_being[df_being["Rank"].between(1, 10)]
    df_types = df_types[df_types["Rank"].between(1, 10)]
    return pd.concat([df_being, df_types], ignore_index=True)

def jaccard_similarity(sets_by_country):
    countries = sorted(sets_by_country.index)
    jacc = pd.DataFrame(0.0, index=countries, columns=countries)
    for i, ci in enumerate(countries):
        si = sets_by_country.loc[ci]
        for cj in countries[i:]:
            sj = sets_by_country.loc[cj]
            inter = len(si & sj)
            union = len(si | sj)
            sim = inter / union if union else 0.0
            jacc.loc[ci, cj] = jacc.loc[cj, ci] = sim
    return jacc

def cosine_similarity_rankpoints(pivot_points):
    A = pivot_points.values.astype(float)
    norms = np.linalg.norm(A, axis=1, keepdims=True)
    norms[norms == 0] = 1.0
    cos = (A @ A.T) / (norms @ norms.T)
    return pd.DataFrame(cos, index=pivot_points.index, columns=pivot_points.index)

def threshold_communities(sim_df, thresh):
    unvisited = set(sim_df.index)
    communities = []
    while unvisited:
        c = unvisited.pop()
        group = {c}
        added = True
        while added:
            added = False
            for x in list(unvisited):
                if any(sim_df.loc[x, g] >= thresh for g in group):
                    group.add(x); unvisited.remove(x); added = True
        communities.append(sorted(group))
    return communities

def lift(a_support, b_support, ab_support, universe):
    # support = count / universe; lift = P(A∩B) / (P(A) P(B))
    if a_support == 0 or b_support == 0:
        return np.nan
    pa = a_support / universe
    pb = b_support / universe
    pab = ab_support / universe
    return pab / (pa * pb) if pa*pb else np.nan

def itemset_mining(country_sets, max_k=3, top_n=30, min_support=2):
    """
    Simple frequent itemset mining up to size k (2 and 3 by default).
    Returns DataFrames for pairs and triples with support and lift.
    """
    countries = list(country_sets.index)
    N = len(countries)
    # 1-item supports
    item_support = Counter()
    for s in country_sets:
        for x in s:
            item_support[x] += 1

    def pairs_stats():
        rows = []
        all_items = sorted(item_support.keys())
        for a, b in combinations(all_items, 2):
            ab = sum(1 for s in country_sets if a in s and b in s)
            if ab >= min_support:
                L = lift(item_support[a], item_support[b], ab, N)
                rows.append((a, b, ab, item_support[a], item_support[b], L))
        df = pd.DataFrame(rows, columns=["A","B","Support_AB","Support_A","Support_B","Lift"])
        df.sort_values(["Support_AB","Lift"], ascending=[False, False], inplace=True)
        return df.head(top_n)

    def triples_stats():
        rows = []
        all_items = sorted(item_support.keys())
        for a, b, c in combinations(all_items, 3):
            abc = sum(1 for s in country_sets if (a in s and b in s and c in s))
            if abc >= min_support:
                # Approx lift 3-way vs independence
                pa, pb, pc = item_support[a]/N, item_support[b]/N, item_support[c]/N
                pabc = abc/N
                L = pabc / (pa*pb*pc) if pa*pb*pc else np.nan
                rows.append((a,b,c,abc,item_support[a],item_support[b],item_support[c],L))
        df = pd.DataFrame(rows, columns=["A","B","C","Support_ABC","Support_A","Support_B","Support_C","Lift3"])
        df.sort_values(["Support_ABC","Lift3"], ascending=[False, False], inplace=True)
        return df.head(top_n)

    pairs = pairs_stats()
    triples = triples_stats() if max_k >= 3 else pd.DataFrame()
    return (pd.DataFrame(item_support.items(), columns=["Predictor","Support_1"]).sort_values("Support_1", ascending=False),
            pairs, triples, N)

def community_signatures(sub, communities, label, out_path):
    """
    For each community (list of countries), compute:
      - top predictors by frequency and by summed Points
      - category mix
      - predictors over-represented vs global (simple risk ratio)
    """
    lines = []
    # global baselines
    global_sets = sub.groupby("Country")["Predictor"].apply(lambda s: set(s.tolist()))
    global_support = Counter()
    for s in global_sets:
        for x in s:
            global_support[x] += 1
    G = len(global_sets)

    for i, group in enumerate(communities, start=1):
        gname = f"{label} Community {i}"
        subg = sub[sub["Country"].isin(group)].copy()

        # Frequency and Points
        freq = subg.groupby("Predictor")["Country"].nunique().sort_values(ascending=False)
        pts  = subg.groupby("Predictor")["Points"].sum().sort_values(ascending=False)

        # Category mix
        subg["Category"] = subg["Predictor"].map(PRED_TO_CATEGORY).fillna("Uncategorized")
        cat_mix = subg.groupby("Category")["Country"].nunique().sort_values(ascending=False)

        # Over-representation vs global: RR = P(x|group)/P(x|global)
        group_sets = subg.groupby("Country")["Predictor"].apply(lambda s: set(s.tolist()))
        H = len(group_sets)
        group_support = Counter()
        for s in group_sets:
            for x in s:
                group_support[x] += 1

        rows = []
        for pred, gs in group_support.items():
            ps_group = gs / H if H else 0.0
            ps_global = (global_support[pred] / G) if G else 0.0
            rr = (ps_group / ps_global) if ps_global else np.nan
            rows.append((pred, gs, global_support[pred], ps_group, ps_global, rr))
        rr_df = (pd.DataFrame(rows, columns=["Predictor","GroupSupport","GlobalSupport","P_inGroup","P_Global","RiskRatio"])
                 .sort_values(["RiskRatio","GroupSupport"], ascending=[False, False])
                 .head(15))

        # Write section
        lines.append(f"### {gname}  (n={H} countries)")
        lines.append(f"Countries: {', '.join(group)}\n")
        lines.append("Top predictors by **frequency** (countries containing in top-10):")
        lines.append(rr_df[["Predictor","GroupSupport","GlobalSupport","RiskRatio"]].to_string(index=False))
        lines.append("\nTop predictors by **points** (rank-weighted):")
        pts_df = pts.reset_index().rename(columns={"index":"Predictor","Points":"TotalPoints"}).head(15)
        lines.append(pts_df.to_string(index=False))
        lines.append("\nCategory mix (countries with ≥1 predictor from category in their top-10):")
        lines.append(cat_mix.to_string())
        lines.append("\n")

    with open(out_path, "a", encoding="utf-8") as f:
        f.write("\n".join(lines) + "\n")

def write_markdown_report(dv, sub, out_path, top1_counts, rank1_lists, in_top10, top10_lists,
                          pairs, triples, jacc_comms, cos_comms):
    lines = []
    lines.append(f"# {dv} — Top-10 Predictor Patterns\n")

    # 1) “Who had what as #1?”
    lines.append("## Predictors that were **#1** and where")
    lines.append(top1_counts.to_string(index=False))
    lines.append("\n### Countries by #1 predictor")
    # Make a compact bullet list
    for _, row in rank1_lists.iterrows():
        pred = row["Predictor"]
        countries = row["Countries_list"]
        lines.append(f"- **{pred}** — {len(countries)} countries: {', '.join(countries)}")
    lines.append("")

    # 2) “Who had what in top-10?”
    lines.append("## Predictors appearing in **top-10** (support across countries)")
    lines.append(in_top10.to_string(index=False))
    lines.append("\n### Countries by top-10 presence")
    for _, row in top10_lists.iterrows():
        pred = row["Predictor"]
        countries = row["Countries_list"]
        lines.append(f"- **{pred}** — {len(countries)} countries: {', '.join(countries)}")
    lines.append("")

    # 3) Pair & trio patterns
    lines.append("## Frequent **pairs** of predictors (support & lift)")
    if not pairs.empty:
        lines.append(pairs.to_string(index=False))
    else:
        lines.append("_No frequent pairs passing the support threshold._")
    lines.append("")

    lines.append("## Frequent **triples** of predictors (support & lift)")
    if not triples.empty:
        lines.append(triples.to_string(index=False))
    else:
        lines.append("_No frequent triples passing the support threshold._")
    lines.append("")

    # 4) Communities
    lines.append("## Country communities (similar top-10 profiles)")
    lines.append("### Jaccard-based (shared set overlap)")
    for i, comm in enumerate(jacc_comms, start=1):
        lines.append(f"- **Community {i}**: {', '.join(comm)}")
    lines.append("")
    lines.append("### Cosine-based (rank-weighted signature)")
    for i, comm in enumerate(cos_comms, start=1):
        lines.append(f"- **Community {i}**: {', '.join(comm)}")
    lines.append("")

    with open(out_path, "w", encoding="utf-8") as f:
        f.write("\n".join(lines) + "\n")

def analyze_outcome(df_all, dv,
                    jacc_thresh=0.5,
                    cos_thresh=0.85,
                    top_pairs=30,
                    top_triples=30,
                    min_support=2):
    sub = df_all[df_all["Type"] == dv].copy()
    if sub.empty:
        print(f"[skip] No rows for {dv}")
        return

    out_path = OUT_DIR / dv
    out_path.mkdir(exist_ok=True)

    # Prep
    sub["Points"] = sub["Rank"].map(POINTS_MAP).astype(int)

    # 1) Rank #1 commonalities 
    top1 = sub[sub["Rank"] == 1][["Country","Predictor"]]
    top1_counts = (top1.groupby("Predictor")["Country"]
                        .nunique()
                        .sort_values(ascending=False)
                        .rename("Countries_with_Rank1")
                        .reset_index())
    top1_counts.to_csv(out_path / "01_rank1_counts.csv", index=False)
    rank1_lists = (top1.groupby("Predictor")["Country"]
                        .apply(lambda s: sorted(s.unique()))
                        .reset_index()
                        .rename(columns={"Country":"Countries_list"}))
    rank1_lists.to_csv(out_path / "02_rank1_country_lists.csv", index=False)

    # 2) Top-10 presence 
    in_top10 = (sub.groupby("Predictor")["Country"]
                    .nunique()
                    .sort_values(ascending=False)
                    .rename("Countries_in_Top10")
                    .reset_index())
    in_top10.to_csv(out_path / "03_in_top10_counts.csv", index=False)
    top10_lists = (sub.groupby("Predictor")["Country"]
                      .apply(lambda s: sorted(s.unique()))
                      .reset_index()
                      .rename(columns={"Country":"Countries_list"}))
    top10_lists.to_csv(out_path / "04_in_top10_country_lists.csv", index=False)

    # 3) Summary w/ ranks, importance, category 
    rank_stats = sub.groupby("Predictor")["Rank"].agg(Avg_Rank="mean", Median_Rank="median").reset_index()
    imp_stats = sub.groupby("Predictor")["Importance"].mean().rename("Mean_Importance").reset_index()
    summary = (top1_counts.merge(in_top10, on="Predictor", how="outer")
                         .merge(rank_stats, on="Predictor", how="outer")
                         .merge(imp_stats, on="Predictor", how="outer"))
    summary["Category"] = summary["Predictor"].map(PRED_TO_CATEGORY).fillna("Uncategorized")
    summary.sort_values(["Countries_with_Rank1","Countries_in_Top10","Avg_Rank"],
                        ascending=[False,False,True], inplace=True)
    summary.to_csv(out_path / "05_predictor_summary.csv", index=False)

    # 4) Similar-country groupings 
    sets = sub.groupby("Country")["Predictor"].apply(lambda s: set(s.tolist()))
    jacc = jaccard_similarity(sets)
    jacc.to_csv(out_path / "06_country_jaccard_top10.csv")

    jacc_comms = threshold_communities(jacc, jacc_thresh)
    pd.DataFrame({"Community_ID": range(1, len(jacc_comms)+1),
                  "Countries": jacc_comms}).to_csv(out_path / "07_country_communities_by_jaccard.csv", index=False)

    pivot_pts = sub.pivot_table(index="Country", columns="Predictor", values="Points", aggfunc="sum", fill_value=0)
    cos = cosine_similarity_rankpoints(pivot_pts)
    cos.to_csv(out_path / "08_country_cosine_rankpoints.csv")

    cos_comms = threshold_communities(cos, cos_thresh)
    pd.DataFrame({"Community_ID": range(1, len(cos_comms)+1),
                  "Countries": cos_comms}).to_csv(out_path / "09_country_communities_by_cosine.csv", index=False)

    # 5) Category-level patterns per country 
    sub["Category"] = sub["Predictor"].map(PRED_TO_CATEGORY).fillna("Uncategorized")
    cat_counts = (sub.groupby(["Country","Category"])["Predictor"].count()
                    .rename("Top10_Count").reset_index())
    cat_wide = cat_counts.pivot(index="Country", columns="Category", values="Top10_Count").fillna(0).astype(int)
    cat_wide.to_csv(out_path / "10_country_category_counts_in_top10.csv")

    # #1 predictor’s category lists
    top1_cat = sub[sub["Rank"]==1].assign(Category=lambda x: x["Predictor"].map(PRED_TO_CATEGORY).fillna("Uncategorized"))
    top1_cat_lists = (top1_cat.groupby("Category")["Country"]
                           .apply(lambda s: sorted(s.unique()))
                           .reset_index()
                           .rename(columns={"Country":"Countries_list"}))
    top1_cat_lists.to_csv(out_path / "11_rank1_category_country_lists.csv", index=False)

    # 6) Frequent itemsets (pairs & triples) with lift 
    support1, pairs, triples, N = itemset_mining(sets, max_k=3, top_n=max(30, len(sets)//2), min_support=2)
    support1.to_csv(out_path / "12_item_support_1.csv", index=False)
    pairs.to_csv(out_path / "13_frequent_pairs.csv", index=False)
    triples.to_csv(out_path / "14_frequent_triples.csv", index=False)

    # 7) Markdown report (bullets) 
    report_md = out_path / "REPORT.md"
    write_markdown_report(dv, sub, report_md, top1_counts, rank1_lists, in_top10, top10_lists,
                          pairs, triples, jacc_comms, cos_comms)

    # 8) Community signatures (over-represented predictors/categories) 
    community_signatures(sub, jacc_comms, "Jaccard", out_path / "REPORT.md")
    community_signatures(sub, cos_comms, "Cosine",  out_path / "REPORT.md")

    print(f"[{dv}] wrote outputs → {out_path}")

if __name__ == "__main__":
    df_all = load_top10()
    # Quick sanity: ensure essential cols exist
    expected_cols = {"Country","Type","Rank","Predictor","Importance"}
    missing = expected_cols - set(df_all.columns)
    if missing:
        raise ValueError(f"Missing columns in input CSVs: {missing}")

    for dv in OUTCOMES:
        analyze_outcome(df_all, dv,
                        jacc_thresh=0.5,   # overlap threshold for set-based communities
                        cos_thresh=0.85,   # similarity threshold for rank-weighted communities
                        top_pairs=30,
                        top_triples=30,
                        min_support=2)

    print(f"Done. See {OUT_DIR.resolve()}")
